In [1]:
import os
import cv2

import matplotlib
import matplotlib.pyplot as plt

import io
import scipy.misc
import numpy as np
from six import BytesIO
from PIL import Image, ImageDraw, ImageFont

import tensorflow as tf

from object_detection.utils import label_map_util
from object_detection.utils import config_util
from object_detection.utils import visualization_utils as viz_utils
from object_detection.builders import model_builder

%matplotlib inline

In [2]:
tf.test.is_gpu_available()

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


False

# Get frames from Video

In [3]:
folder = '../data/raw/videos/getty_tracking'

In [4]:
folder_out = '../data/raw/images/getty_tracking'

In [ ]:
for video_file in 095444444444444444444444444

    video = cv2.VideoCapture(os.path.join(folder, video_file))
    count = 0;
    while video.isOpened():
        try:
            # Read video capture
            ret, frame = video.read()
            # Display each frame
            cv2.imshow("video", frame)
            # show one frame at a time
            key = cv2.waitKey(0)
            while key not in [ord('q'), ord('s'), ord('c')]:
                key = cv2.waitKey(0)
            # Quit when 'q' is pressed
            if key == ord('q'):
                break
            # Manual save when 'q' is pressed
            if key == ord('s'):
                cv2.imwrite(os.path.join(folder_out, f"{video_file}_frame{count}.jpg"), frame)
            # Continue and save one frame every ten frames when 'c' is pressed
            if key == ord('c'):
                if divmod(count, nframes)[1] == 0:
                    print(cv2.imwrite(os.path.join(folder_out, f"{video_file}_frame{count}.jpg"), frame))
            count += 1
        except:
            # Release capture objectccccccccccc
            video.release()
            # Exit and distroy all windows
            cv2.destroyAllWindows()
    cv2.destroyAllWindows()

In [5]:
video.release()

# Using package detection on a frame

In [3]:
def load_image_into_numpy_array(path):
  """Load an image from file into a numpy array.

  Puts image into numpy array to feed into tensorflow graph.
  Note that by convention we put it into a numpy array with shape
  (height, width, channels), where channels=3 for RGB.

  Args:
    path: the file path to the image

  Returns:
    uint8 numpy array with shape (img_height, img_width, 3)
  """
  img_data = tf.io.gfile.GFile(path, 'rb').read()
  image = Image.open(BytesIO(img_data))
  (im_width, im_height) = image.size
  return np.array(image.getdata()).reshape(
      (im_height, im_width, 3)).astype(np.uint8)

def get_keypoint_tuples(eval_config):
  """Return a tuple list of keypoint edges from the eval config.
  
  Args:
    eval_config: an eval config containing the keypoint edges
  
  Returns:
    a list of edge tuples, each in the format (start, end)
  """
  tuple_list = []
  kp_list = eval_config.keypoint_edge
  for edge in kp_list:
    tuple_list.append((edge.start, edge.end))
  return tuple_list

In [4]:
#This must be changed by new user
pipeline_config = "../object_detection_API/exported-models/package_ssd_mobilenet_v2_fpnlite_320/pipeline.config"
model_dir = "../object_detection_API/exported-models/package_ssd_mobilenet_v2_fpnlite_320/checkpoint/"

# Load pipeline config and build a detection model
configs = config_util.get_configs_from_pipeline_file(pipeline_config)
model_config = configs['model']
detection_model = model_builder.build(
      model_config=model_config, is_training=False)

# Restore checkpoint
ckpt = tf.compat.v2.train.Checkpoint(
      model=detection_model)
ckpt.restore(os.path.join(model_dir, 'ckpt-0')).expect_partial()

def get_model_detection_function(model):
  """Get a tf.function for detection."""

  @tf.function
  def detect_fn(image):
    """Detect objects in image."""

    image, shapes = model.preprocess(image)
    prediction_dict = model.predict(image, shapes)
    detections = model.postprocess(prediction_dict, shapes)

    return detections, prediction_dict, tf.reshape(shapes, [-1])

  return detect_fn

detect_fn = get_model_detection_function(detection_model)

In [5]:
label_map_path = "../object_detection_API/labelmap.pbtxt"
label_map = label_map_util.load_labelmap(label_map_path)
categories = label_map_util.convert_label_map_to_categories(
    label_map,
    max_num_classes=label_map_util.get_max_label_map_index(label_map),
    use_display_name=True)
category_index = label_map_util.create_category_index(categories)
label_map_dict = label_map_util.get_label_map_dict(label_map, use_display_name=True)

In [6]:
image_path = "../data/raw/images/getty_tracking/gettyimages-162812919-640_adpp.mp4_frame0.jpg"
image_np = load_image_into_numpy_array(image_path)

In [7]:
plt.imshow(image_np)

<Figure size 432x288 with 1 Axes>
[imagen eliminada en drive_snapshot: fotograma de stock de Getty]

In [8]:
input_tensor = tf.convert_to_tensor(np.expand_dims(image_np, 0), dtype=tf.float32)
detections, predictions_dict, shapes = detect_fn(input_tensor)

In [10]:
label_id_offset = 1
image_np_with_detections = image_np.copy()

# Use keypoints if available in detections
keypoints, keypoint_scores = None, None
if 'detection_keypoints' in detections:
    keypoints = detections['detection_keypoints'][0].numpy()
    keypoint_scores = detections['detection_keypoint_scores'][0].numpy()

viz_utils.visualize_boxes_and_labels_on_image_array(
      image_np_with_detections,
      detections['detection_boxes'][0].numpy(),
      (detections['detection_classes'][0].numpy() + label_id_offset).astype(int),
      detections['detection_scores'][0].numpy(),
      category_index,
      use_normalized_coordinates=True,
      max_boxes_to_draw=200,
      min_score_thresh=.5,
      agnostic_mode=False,
      keypoints=keypoints,
      keypoint_scores=keypoint_scores,
      keypoint_edges=get_keypoint_tuples(configs['eval_config']))

plt.figure(figsize=(12,16))
plt.imshow(image_np_with_detections)
plt.show()

<Figure size 864x1152 with 1 Axes>
[imagen eliminada en drive_snapshot: fotograma de stock de Getty]

# Naive Box Tracking

In [16]:
def iou(box1, box2):
    # Assign variable names to coordinates for clarity
    (box1_x1, box1_y1, box1_x2, box1_y2) = box1
    (box2_x1, box2_y1, box2_x2, box2_y2) = box2
    
    # Calculate the (yi1, xi1, yi2, xi2) coordinates of the intersection of box1 and box2. Calculate its Area.
    xi1 = max(box1_x1, box2_x1)
    yi1 = max(box1_y1, box2_y1)
    xi2 = min(box1_x2, box2_x2)
    yi2 = min(box1_y2, box2_y2)
    inter_width = max(xi2 - xi1, 0)
    inter_height = max(yi2 - yi1, 0)
    inter_area = inter_width*inter_height  

    # Calculate the Union area by using Formula: Union(A,B) = A + B - Inter(A,B)
    box1_area = (box1_x2-box1_x1)*(box1_y2-box1_y1)
    box2_area = (box2_x2-box2_x1)*(box2_y2-box2_y1)
    union_area = box1_area + box2_area - inter_area
    
    # compute the IoU
    iou = inter_area/union_area
    
    return iou

In [17]:
def detection_mask(detections):
    correct_high = detections['detection_boxes'][0][:,2] - detections['detection_boxes'][0][:,0] < 2/3
    correct_width = detections['detection_boxes'][0][:,3] - detections['detection_boxes'][0][:,1] < 2/3
    correct_score = detections['detection_scores'][0] > 0.5
    msk = correct_high & correct_width & correct_score
    return msk

In [50]:
image_path = "../data/raw/images/getty_tracking/gettyimages-162812919-640_adpp.mp4_frame0.jpg"
image_np = load_image_into_numpy_array(image_path)

In [19]:
input_tensor = tf.convert_to_tensor(np.expand_dims(image_np, 0), dtype=tf.float32)
detections, predictions_dict, shapes = detect_fn(input_tensor)

In [29]:
detection_mask(detections)

<tf.Tensor: shape=(100,), dtype=bool, numpy=
array([ True,  True,  True, False,  True, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False])>

In [38]:
detections.keys()

dict_keys(['detection_boxes', 'detection_scores', 'detection_classes', 'num_detections', 'raw_detection_boxes', 'raw_detection_scores', 'detection_multiclass_scores', 'detection_anchor_indices'])

In [47]:
detections['detection_anchor_indices']

<tf.Tensor: shape=(1, 100), dtype=int32, numpy=
array([[ 2604,  1558,  3907, 12459,  1410,  7122,  2577, 11948, 12673,
         2778, 11978,  2539,  2299,  2355,  3763,  2325, 11961,  3524,
         8919,  9009,  4283,  2313,  9807, 12524, 10720, 10160, 12468,
         2033,   663,  2082,  3676,  2349,  1889,  9291,  1178,  4973,
         9519, 10584,  7112, 10184,  7092,  2793,   916, 12507,  9147,
         9033,  2066,  6970, 10659,  2374, 11852,  2122,  2540,  9470,
         3282,   935, 10287,  9237,  1400, 11936,  2617,  3662,  4642,
         1182, 11946,  1545,  2158,  7533,  1079,  1836, 11982,  2592,
         2072,  3711,  2112,  1844,  2108,  2316,  3064,  7756,  2062,
         7357,  2088,  9524,  3070, 11970,  1832,  9069,  9482,  2542,
         1649,  3435,  1826, 12701, 10476,  2294,  1156,  4161,  2326,
         9699]], dtype=int32)>

In [34]:
path = "../data/raw/images/getty_tracking/"
_, _, filenames = next(os.walk(path))

In [49]:
label_id_offset = 1
image_np_with_detections = image_np.copy()

# Use keypoints if available in detections
keypoints, keypoint_scores = None, None
if 'detection_keypoints' in detections:
    keypoints = detections['detection_keypoints'][0].numpy()
    keypoint_scores = detections['detection_keypoint_scores'][0].numpy()

viz_utils.visualize_boxes_and_labels_on_image_array(
      image_np_with_detections,
      detections['detection_boxes'][0][msk].numpy(),
      (detections['detection_classes'][0][msk].numpy() + label_id_offset).astype(int),
      detections['detection_scores'][0][msk].numpy(),
      category_index,
      track_ids=np.array([0, 1, 2, 3]),
      use_normalized_coordinates=True,
      max_boxes_to_draw=200,
      min_score_thresh=.5,
      agnostic_mode=False,
      keypoints=keypoints,
      keypoint_scores=keypoint_scores,
      keypoint_edges=get_keypoint_tuples(configs['eval_config']))

plt.figure(figsize=(12,16))
plt.imshow(image_np_with_detections)
plt.show()

<Figure size 864x1152 with 1 Axes>
[imagen eliminada en drive_snapshot: fotograma de stock de Getty]